# gradio_main.py — Overview and Flow

This notebook documents the purpose and flow of `app/gradio_main.py` (the demo Gradio Student + Librarian UI) and its associated helpers. It explains the key functions, in-memory stores, the student → librarian lifecycle, testing tips, and recommended next steps for refactoring and testing.

Notes: this notebook is *documentation-first* — code snippets are safe to run in a development environment but the main Gradio UI requires the `gradio` package if you want to run the interactive app locally. The test harness in `tests/test_book_request_flow.py` shows how to import the working copy by path without installing Gradio.

## High-level architecture

- Single-file demo: `app/gradio_main.py` (and a working copy `app/gradio_main copy.py`).
- Purpose: provide a minimal demo of a Student-facing request flow and a Librarian console to approve/reject requests, with in-memory stores for rapid iteration.
- Key pieces:
  - In-memory stores: `BOOK_DB`, `BOOK_REQUESTS`, `LIBRARIAN_QUEUES`
  - Student handlers: `submit_book_request`, `refresh_my_requests`
  - Librarian handlers: `librarian_list_pending`, `librarian_approve`, `librarian_reject`
  - UI builders: `build_student_ui`, `build_librarian_ui`, and `build_demo` (the Blocks UI).
- Demo-friendly stubs: light-weight LLM/search/RAG stubs are included so the modules import even if heavy deps are absent.

## Key functions and their purpose

- `_book_label(book_id) -> str` — returns a deterministic, human-readable label for a given `book_id` (title, author, lexile, grade, and the id in parentheses).
- `_parse_id_from_label(label) -> str` — extracts the `book_id` from a label (used when dropdowns show pretty labels).
- `_student_requests(username) -> list` — helper that returns the list of requests for a given student from the `BOOK_REQUESTS` store.

Student-side handlers
- `submit_book_request(user, book_choice, date_needed, special) -> (message, rows, approved_choices)` — validates inputs, avoids duplicate pending requests, appends a pending request to `BOOK_REQUESTS` and `LIBRARIAN_QUEUES`, and returns a message plus table rows and a list of approved choices for the student's UI.
- `refresh_my_requests(user) -> (rows, approved_choices)` — builds the table rows and the approved-request dropdown choices from the student's requests.

Librarian-side handlers
- `librarian_list_pending() -> list` — returns formatted labels for pending items in `LIBRARIAN_QUEUES[
]` suitable for a dropdown.
- `librarian_approve(selected_label, available_on, note) -> (message, pending_list, student_rows)` — finds the selected pending item, marks the student request as `approved`, updates the queue entry, and returns an updated pending list and the student's refreshed rows.
- `librarian_reject(selected_label, note) -> (message, pending_list, student_rows)` — similar to approve but marks the request `rejected` and updates student rows.

UI builders
- `build_student_ui()` and `build_librarian_ui()` return component references and attach event wiring (safely: optional callbacks are only attached when callable so missing helpers don't raise at import time).
- `build_demo(return_blocks=False)` constructs the full Gradio `Blocks` UI and can return the Blocks object for mounting or testing.

Other helpers
- `route_recency_aware`, `ui_librarian_book_prompt`, and `ui_rag_upload` — router and RAG upload utilities present in the file; demo stubs are used by default when legacy features are disabled.

In [ ]:
# Example: safe import of the working copy by file path (useful when the module name isn't a valid identifier)
# This pattern is used by tests/test_book_request_flow.py.
import importlib.util, os, sys
root = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
mod_path = os.path.join(root, 'app', 'gradio_main copy.py')
spec = importlib.util.spec_from_file_location('gradio_copy', mod_path)
gr_mod = importlib.util.module_from_spec(spec)
# If you want to run tests without installing real 'gradio', inject a tiny dummy module:
if 'gradio' not in sys.modules:
    import types
    dummy = types.ModuleType('gradio')
    # provide minimal symbols used at import-time (Blocks, Accordion, etc.)
    class _Ctx:
        def __enter__(self):
            return self
        def __exit__(self, exc_type, exc, tb):
            return False
    for name in ['Blocks','Accordion','Tabs','Tab','Row']:
        setattr(dummy, name, lambda *a, **k: _Ctx())
    for name in ['Slider','Dataframe','Markdown','Dropdown','Textbox','Button','CheckboxGroup','Files','Code','Plot','Radio','Checkbox']:
        setattr(dummy, name, lambda *a, **k: _Ctx())
    sys.modules['gradio'] = dummy

spec.loader.exec_module(gr_mod)
print('Imported module:', getattr(gr_mod, '__file__', '<built>'))

In [ ]:
# Compact demo-safe lifecycle simulation (pseudo-code)
try:
    from app import gradio_main as gm
    print('Using app.gradio_main stores')
    BOOK_REQUESTS = getattr(gm, 'BOOK_REQUESTS', {})
    submit = getattr(gm, 'submit_book_request')
    list_pending = getattr(gm, 'librarian_list_pending')
    approve = getattr(gm, 'librarian_approve')
    refresh = getattr(gm, 'refresh_my_requests')
except Exception as e:
    print('app.gradio_main not available; using local demo stores:', e)
    BOOK_REQUESTS = {}
    _id = 1
    def submit(user, title, date_needed, special=''):
        nonlocal _id
        rid = str(_id); _id += 1
        BOOK_REQUESTS[rid] = {'id':rid,'student':user,'book_id':rid,'book_label':title,'date_needed':date_needed,'special':special,'status':'pending'}
        return ('submitted', [BOOK_REQUESTS[rid]], [])
    def list_pending():
        return [v for v in BOOK_REQUESTS.values() if v['status']=='pending']
    def approve(librarian, selected_id, available_on=None, note=''):
        r = BOOK_REQUESTS.get(selected_id)
        if not r:
            return ('not found', list_pending(), [])
        r['status']='approved'; r['approved_by']=librarian; r['available_on']=available_on; r['note']=note
        return ('approved', list_pending(), [r])
    def refresh(user):
        return ([v for v in BOOK_REQUESTS.values() if v['student']==user], [])

# Simulate: student submits, librarian approves, student refreshes, student returns book
student = 'alice@example.com'
msg, rows, approved = submit(student, 'The Time Machine', '2025-10-01')
print('Submit ->', msg)
print('Pending now:', list_pending())
# Librarian approves the first pending id
pending = list_pending()
if pending:
    pid = pending[0]['id'] if isinstance(pending[0], dict) else pending[0]
    amsg, pend_after, student_rows = approve('lib_head', pid, '2025-10-05', 'Available on request')
    print('Approve ->', amsg)
    print('Pending after approve:', pend_after)
    print('Student rows after approve:', student_rows)
    # Student returns the book (pseudo-check pass)
    passed = True
    if passed:
        BOOK_REQUESTS[pid]['status']='completed'
        BOOK_REQUESTS[pid]['completed_at']='2025-10-07'
        print('Book return logged; completed')
else:
    print('No pending requests in demo')

## Example flow (student → librarian) — step-by-step

This section traces the lifecycle of a student book request through the demo system, from submission to librarian action and final return. It also includes notes about idempotency and UI updates.

1) Student submits a request via `submit_book_request(user, book_label, date_needed, special)`
   - The function extracts the `book_id` using `_parse_id_from_label(book_label)` so the UI can display pretty labels but the backend stores canonical ids.
   - It validates the `book_id` exists in `BOOK_DB`. If not found, it returns an error message to the student UI.
   - It checks for duplicate pending requests by the same student for the same book to avoid spam; if a duplicate exists it returns a helpful message.
   - On success it creates a request object with fields: `{id, student, book_id, book_label, date_needed, special, status:'pending', created_at}` and stores it in `BOOK_REQUESTS` and appends an entry in `LIBRARIAN_QUEUES` for librarian visibility.
   - It returns `(message, student_rows, approved_choices)` where `student_rows` is a table-friendly list and `approved_choices` are labels for any approved items the student can act on.

2) Librarian views pending requests via `librarian_list_pending()`
   - This helper reads `LIBRARIAN_QUEUES` and formats labels using `_book_label(book_id)` so the librarian sees human-friendly descriptions but the label still encodes the internal id.
   - The UI shows the pending list in a dropdown or accordion where each entry maps to one request id.

3) Librarian approves or rejects an item using `librarian_approve(selected_label, available_on, note)` or `librarian_reject(selected_label, note)`
   - The function parses the `request_id` from `selected_label` via `_parse_id_from_label` and looks up the request in `BOOK_REQUESTS`.
   - On approve: it sets `status='approved'`, records `available_on` and optional `note`, and updates the queue entry; it may also post an audit log in a lightweight `AUDIT_LOG` store.
   - On reject: it sets `status='rejected'`, stores the rejection `note`, and updates both `BOOK_REQUESTS` and `LIBRARIAN_QUEUES` accordingly.
   - Both functions return `(message, pending_list, student_rows)` so the UI can refresh the librarian pending dropdown and the affected student's table rows.

4) Student sees their requests update via `refresh_my_requests(user)`
   - This function reads `BOOK_REQUESTS` for the given user and builds the table rows and any approved-choice dropdown items.
   - It is designed to be called after actions that change state (submission, approval, rejection) so the student's UI always reflects the current status without a full page refresh.

5) Student returns a finished book via `log_finished_book(user, approved_label, notes)` (return-with-quiz flow)
   - The function locates the approved request (extract id from `approved_label`), optionally runs a small qualifying quiz (e.g., check short-answer by simple keyword match or call Hermes3 for a short evaluation), and if the student passes, marks the request as `completed` and records `completed_at` and `completion_notes`.
   - The return action increments campaign/leaderboard counters (if applicable) and may trigger a campaign evaluation job (weekly winner selection).

Idempotency and concurrency notes:
- All handlers validate inputs and check current status before mutating stores to avoid race conditions (e.g., approving an already-approved request should return a friendly message, not error).
- The in-memory demo stores are single-process and not durable. For production, persist to a database and add transactional updates or optimistic locking.

UI wiring notes (how the Gradio components are connected):
- `submit_book_request` is wired to a student `Button.click` and updates a `Dataframe` and a `Dropdown` in the student tab via the returned tuple.
- `librarian_approve` and `librarian_reject` are wired to librarian buttons and each return updated lists for the librarian dropdown and the affected student's rows so both views stay consistent.

The code cell earlier in this notebook provides a compact pseudo-code simulation of the lifecycle that you can run (it is guarded and demo-safe).